# Flatten extracted_data.json to CSV

One row per unit; quote-level colour_in/colour_out spread to each unit.

**Formatting applied:**
- **Colors:** Dimension tokens (e.g. `38-5/8 x 35-1/8`) stripped from colour strings; any `color_in` or `color_out` that is not "WHT" is mapped to "COLOR".
- **Window description:** " LEFT" / " RIGHT" removed; "CASMENT" → "CASEMENT"; "double end slider" → "DOUBLE SLIDER". Units with "4-9/16" in description are skipped.
- **Excluded types:** Rows with window type "small fixed", "sealed unit", "half round", or "dummy window" are dropped.
- **Unit cost:** Looked up from price list by sq_ft and window type; white vs colour tiers by `color_in` (WHT = white). Columns: `quote_unit_price` (quote’s base price), `pricebook_unit_price` (from price list), `sq_ft_bucket` (e.g. 1-6, 6-9, 9-12, 12+).

## Setup
Imports and file paths.

In [135]:
import json
import re
from pathlib import Path
import pandas as pd
import yaml

DATA_PATH = Path("2026/parsed_all_vp_quotes_2026.json")
# DATA_PATH = Path("pre2026 Quotes/extracted_data.json")
OUT_PATH = Path("extracted_units_2026.csv")
# Price list for unit cost lookup (YAML from parse_price_list_pdf)
PRICE_LIST_PATH = Path("price_list_extracted_2026.yaml")

## Load data
Read the extracted quote lines from JSON.

In [136]:
with open(DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)
print(f"Loaded {len(data)} quote lines")

Loaded 116 quote lines


## Helper: strip dimensions from color strings
Remove measurement tokens (e.g. `38-5/8 x 35-1/8`) so only the color remains (e.g. WHT, BLACK 525).

In [137]:
def drop_measurements(s):
    """Strip dimension tokens; keep only color (e.g. WHT, BLACK 525, CREAM L/G 492, IRON ORE 5P6)."""
    if not s or not isinstance(s, str):
        return "" if s is None else str(s)
    s = s.strip()
    if not s:
        return ""
    def is_dim(t):
        if t.lower() == "x":
            return True
        if re.match(r"^\d+-\d+/\d+$", t):
            return True
        if re.match(r"^\d+\.\d+$", t):
            return True
        return False
    parts = s.split()
    color_parts = [p for p in parts if not is_dim(p)]
    while color_parts and re.match(r"^\d+$", color_parts[0]):
        color_parts.pop(0)
    return " ".join(color_parts).strip()

## Flatten to one row per unit
Loop over quote lines and units; spread `colour_in` / `colour_out` to each unit. Use `description` as `window_description`.

In [138]:
rows = []
for line in data:
    quote_id = line.get("quote_id", "")
    source_file = line.get("source_file", "")
    raw_in = line.get("colour_in") or line.get("color_in") or ""
    raw_out = line.get("colour_out") or line.get("color_out") or ""
    color_in = drop_measurements(raw_in)
    color_out = drop_measurements(raw_out)
    if color_in and color_in.strip().upper() != "WHT":
        color_in = "COLOR"
    if color_out and color_out.strip().upper() != "WHT":
        color_out = "COLOR"
    for unit in line.get("units") or []:
        window_description = (unit.get("description", "") or "").strip()
        if "4-9/16" in window_description:
            continue
        window_description = window_description.replace(" LEFT", "").replace(" RIGHT", "").replace("CASMENT", "CASEMENT").strip()
        rows.append({
            "quote_id": quote_id,
            "window_description": window_description,
            "width_in": unit.get("width_in", ""),
            "height_in": unit.get("height_in", ""),
            "sq_ft": unit.get("sq_ft", ""),
            "base_price": unit.get("base_price", ""),
            "color_in": color_in,
            "color_out": color_out,
            "source_file": source_file,
        })
print(f"Built {len(rows)} unit rows")

Built 104 unit rows


## Build DataFrame
Convert the list of row dicts to a pandas DataFrame.

In [139]:
df = pd.DataFrame(rows)
df.head(10)

,quote_id,window_description,width_in,height_in,sq_ft,base_price,color_in,color_out,source_file
0,519307,CASEMENT,22.6875,69.0000,11.67,331.53,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
1,519307,VINYL FIXED,22.6875,69.0000,11.67,278.56,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
2,519307,HALF ROUND,57.3750,28.6875,12.08,263.14,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
3,519307,VINYL FIXED,57.3750,70.9375,29.00,692.23,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
4,519307,VINYL FIXED,22.6875,69.0000,11.67,278.56,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
5,519307,CASEMENT,22.6875,69.0000,11.67,331.53,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
6,519310,VINYL FIXED,22.6250,56.8750,9.67,170.94,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
7,519310,CASEMENT,22.6250,56.8750,9.67,251.84,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
8,519310,CASEMENT,22.6250,56.8750,9.67,251.84,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
9,519310,VINYL FIXED,22.6250,56.8750,9.67,170.94,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF


In [140]:
df['window_description'].value_counts()

window_description
CASEMENT             48
VINYL FIXED          28
SINGLE SLIDER        13
HALF ROUND            7
SMALL FIXED           3
DUMMY WINDOW          2
DOUBLE END SLIDER     1
DOUBLE SLIDER         1
SEALED UNIT           1
Name: count, dtype: int64

In [141]:
# Map any description containing DOUBLE and SLIDER -> DOUBLE SLIDER; drop small_fixed, sealed unit, half round, dummy window
df["window_description"] = df["window_description"].str.replace(
    r"(?i)(?=.*\bdouble\b)(?=.*\bslider\b).*", "DOUBLE SLIDER", regex=True
)
drop_types = {"small fixed", "sealed unit", "half round", "dummy window"}
mask = df["window_description"].str.lower().str.strip().isin(drop_types)
df = df.loc[~mask].reset_index(drop=True).copy()
df.head(10)

,quote_id,window_description,width_in,height_in,sq_ft,base_price,color_in,color_out,source_file
0,519307,CASEMENT,22.6875,69.0000,11.67,331.53,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
1,519307,VINYL FIXED,22.6875,69.0000,11.67,278.56,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
2,519307,VINYL FIXED,57.3750,70.9375,29.00,692.23,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
3,519307,VINYL FIXED,22.6875,69.0000,11.67,278.56,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
4,519307,CASEMENT,22.6875,69.0000,11.67,331.53,COLOR,COLOR,PO#_14FIELDING_Order_519307.PDF
5,519310,VINYL FIXED,22.6250,56.8750,9.67,170.94,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
6,519310,CASEMENT,22.6250,56.8750,9.67,251.84,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
7,519310,CASEMENT,22.6250,56.8750,9.67,251.84,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
8,519310,VINYL FIXED,22.6250,56.8750,9.67,170.94,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF
9,519310,VINYL FIXED,22.6250,56.8750,9.67,170.94,WHT,COLOR,PO#_14FIELDING_Order_519310.PDF


## Unit cost from price list

Look up cost per unit using `sq_ft` and `window_description` from the price list YAML. Use white tiers when `color_in` is WHT, otherwise use colour tiers. If `sq_ft` exceeds the highest tier's `max_sf`, use `sq_ft * per_sf_rate`.

In [142]:
with open(PRICE_LIST_PATH, encoding="utf-8") as f:
    price_list = yaml.safe_load(f)

# Map flattened window_description to price list key (try exact then 4_9_16_ prefix)
DESC_TO_KEY = {
    "VINYL FIXED": "fixed_casement",
    "FIXED CASEMENT": "fixed_casement",
    "PICTURE WINDOW": "picture_window",
    "SINGLE SLIDER": "single_slider_tilt_out",
    "DOUBLE SLIDER": "double_slider_tilt_out",
    "SINGLE HUNG": "single_hung_tilt",
    "DOUBLE HUNG": "double_hung_tilt",
    "CASEMENT": "casement",
    "AWNING": "awning",
}

def _price_list_key(desc, pl):
    """Resolve window_description to a price list key. Tries DESC_TO_KEY, then normalized snake_case, then 4_9_16_ prefix. Returns None if desc is falsy."""
    if not desc:
        return None
    d = str(desc).strip()
    if d in DESC_TO_KEY and DESC_TO_KEY[d] in pl:
        return DESC_TO_KEY[d]
    # Any description containing both DOUBLE and SLIDER maps to double_slider_tilt_out
    u = d.upper()
    if "DOUBLE" in u and "SLIDER" in u and "double_slider_tilt_out" in pl:
        return "double_slider_tilt_out"
    norm = d.lower().replace(" ", "_").replace("-", "_")
    if norm in pl:
        return norm
    if "4_9_16_" + norm in pl:
        return "4_9_16_" + norm
    return norm

def _tier_price(tiers, sq_ft):
    """Return the price for sq_ft from a list of tier dicts (max_sf, price, per_sf_rate). Uses the first tier where sq_ft <= max_sf; if above all tiers, returns sq_ft * per_sf_rate. Returns None if no tier applies."""
    if not tiers or sq_ft is None:
        return None
    tiers = [t for t in tiers if isinstance(t, dict)]
    # tier with max_sf that sq_ft falls under
    for t in sorted([t for t in tiers if "max_sf" in t], key=lambda x: x["max_sf"]):
        if sq_ft <= t["max_sf"] and "price" in t:
            return t["price"]
    # above all tiers: use per_sf_rate
    for t in tiers:
        if "per_sf_rate" in t:
            return round(sq_ft * t["per_sf_rate"], 2)
    return None

def _tier_bucket(tiers, sq_ft):
    """Return the tier bucket label for sq_ft (e.g. '1-6', '6-9', '9-12', '12+') from a list of tier dicts. Returns None if tiers are empty or sq_ft is None."""
    if not tiers or sq_ft is None:
        return None
    tiers = [t for t in tiers if isinstance(t, dict) and "max_sf" in t]
    if not tiers:
        return None
    max_sfs = sorted(t["max_sf"] for t in tiers)
    prev = 0
    for m in max_sfs:
        if sq_ft <= m:
            return f"{prev}-{m}" if prev else f"1-{m}"
        prev = m
    return f"{max_sfs[-1]}+"

def unit_cost_from_price_list(row, pl):
    """Look up unit cost and tier bucket for a dataframe row from price list pl. Uses white tiers when color_in is WHT, otherwise colour tiers. Returns (cost, bucket) or (None, None) if window type not in price list."""
    key = _price_list_key(row.get("window_description"), pl)
    if key is None or key not in pl:
        return (None, None)
    entry = pl[key]
    is_white = (str(row.get("color_in") or "").strip().upper() == "WHT")
    if is_white and "white" in entry:
        tiers = entry["white"]
    elif "colour" in entry:
        tiers = entry["colour"]
    else:
        tiers = entry.get("white", entry.get("interior", []))
    sq_ft = row.get("sq_ft")
    cost = _tier_price(tiers, sq_ft)
    bucket = _tier_bucket(tiers, sq_ft)
    return (cost, bucket)

results = df.apply(lambda row: unit_cost_from_price_list(row, price_list), axis=1)
df["unit_cost"] = [r[0] for r in results]
df["tier_bucket"] = [r[1] for r in results]
df = df.rename(columns={"base_price": "quote_unit_price", "unit_cost": "pricebook_unit_price", "tier_bucket": "sq_ft_bucket"})
print(f"Looked up pricebook_unit_price from {PRICE_LIST_PATH.name}; {df['pricebook_unit_price'].notna().sum()} of {len(df)} matched")
df[["window_description", "sq_ft", "color_in", "quote_unit_price", "pricebook_unit_price", "sq_ft_bucket"]].head(15)

Looked up pricebook_unit_price from price_list_extracted_2026.yaml; 91 of 91 matched


,window_description,sq_ft,color_in,quote_unit_price,pricebook_unit_price,sq_ft_bucket
0,CASEMENT,11.67,COLOR,331.53,343.20,9-12
1,VINYL FIXED,11.67,COLOR,278.56,288.37,7+
2,VINYL FIXED,29.00,COLOR,692.23,716.59,7+
3,VINYL FIXED,11.67,COLOR,278.56,288.37,7+
4,CASEMENT,11.67,COLOR,331.53,343.20,9-12
5,VINYL FIXED,9.67,WHT,170.94,176.96,7+
6,CASEMENT,9.67,WHT,251.84,260.70,9-12
7,CASEMENT,9.67,WHT,251.84,260.70,9-12
8,VINYL FIXED,9.67,WHT,170.94,176.96,7+
9,VINYL FIXED,9.67,WHT,170.94,176.96,7+


## Price factor and color-out validation

1. **Price factor (pricebook_unit_price vs quote_unit_price):** Filter to rows where `color_out == "WHT"` so quote price is white-on-white. Compute price factor to verify how the price list’s `pricebook_unit_price` relates to the quote’s `quote_unit_price` (e.g. consistent multiplier).

2. **Color-out add-on %:** Using the same white `color_out` units as a baseline, compare with units where `color_out` is not WHT to estimate what % add-on the exterior/colour option adds (colour premium).

In [143]:
df_wht = df[df['color_out'] == "WHT"].copy()
df_wht['price_factor'] = 1 - (df_wht['quote_unit_price'] / df_wht['pricebook_unit_price'])
df_wht[['window_description', 'sq_ft', 'color_in', 'color_out', 'quote_unit_price', 'pricebook_unit_price', 'price_factor', 'source_file']].sort_values(['window_description', 'color_in'])

,window_description,sq_ft,color_in,color_out,quote_unit_price,pricebook_unit_price,price_factor,source_file
35,CASEMENT,8.03,WHT,WHT,193.62,233.69,0.171466,PO#_161SPROULE_Order_519243.PDF
51,CASEMENT,12.08,WHT,WHT,211.70,262.98,0.194996,PO#_276BLACKTH_Order_519367.PDF
56,CASEMENT,7.50,WHT,WHT,188.12,233.69,0.195002,PO#_504BUSH_Order_518690.PDF
57,CASEMENT,7.50,WHT,WHT,188.12,233.69,0.195002,PO#_504BUSH_Order_518690.PDF
58,CASEMENT,7.50,WHT,WHT,188.12,233.69,0.195002,PO#_504BUSH_Order_518690.PDF
62,CASEMENT,9.67,WHT,WHT,209.86,260.70,0.195013,PO#_7FITGERALD_Order_518771.PDF
63,CASEMENT,8.89,WHT,WHT,188.12,233.69,0.195002,PO#_7FITGERALD_Order_518771.PDF
65,CASEMENT,8.89,WHT,WHT,188.12,233.69,0.195002,PO#_7FITGERALD_Order_518771.PDF
66,CASEMENT,11.56,WHT,WHT,209.86,260.70,0.195013,PO#_7FITGERALD_Order_518771.PDF
68,CASEMENT,9.39,WHT,WHT,209.86,260.70,0.195013,PO#_7FITGERALD_Order_518771.PDF


In [144]:
price_factor = df_wht['price_factor'].mean()
price_factor

np.float64(0.19438365957294598)

In [145]:
df_color = df[df["color_out"] != "WHT"].copy()
df_color["color_out_factor"] = df_color["quote_unit_price"] / (df_color['pricebook_unit_price'] * (1-price_factor))
df_color[['window_description', 'sq_ft', 'color_in','quote_unit_price', 'pricebook_unit_price', 'color_out_factor', 'source_file']].sort_values(['window_description', 'color_in'])

,window_description,sq_ft,color_in,quote_unit_price,pricebook_unit_price,color_out_factor,source_file
0,CASEMENT,11.67,COLOR,331.53,343.20,1.199078,PO#_14FIELDING_Order_519307.PDF
4,CASEMENT,11.67,COLOR,331.53,343.20,1.199078,PO#_14FIELDING_Order_519307.PDF
6,CASEMENT,9.67,WHT,251.84,260.70,1.199100,PO#_14FIELDING_Order_519310.PDF
7,CASEMENT,9.67,WHT,251.84,260.70,1.199100,PO#_14FIELDING_Order_519310.PDF
10,CASEMENT,9.67,WHT,251.84,260.70,1.199100,PO#_14FIELDING_Order_519310.PDF
11,CASEMENT,8.86,WHT,225.75,233.69,1.199111,PO#_14FIELDING_Order_519310.PDF
13,CASEMENT,8.86,WHT,225.75,233.69,1.199111,PO#_14FIELDING_Order_519310.PDF
15,CASEMENT,8.86,WHT,225.75,233.69,1.199111,PO#_14FIELDING_Order_519310.PDF
16,CASEMENT,10.69,WHT,251.84,260.70,1.199100,PO#_14FIELDING_Order_519310.PDF
18,CASEMENT,10.69,WHT,251.84,260.70,1.199100,PO#_14FIELDING_Order_519310.PDF


In [146]:
color_out_factor = df_color['color_out_factor'].mean() - 1
color_out_factor
(1 + color_out_factor * (df['color_out'] == "WHT").astype(int))

0     1.000000
1     1.000000
2     1.000000
3     1.000000
4     1.000000
        ...   
86    1.196728
87    1.196728
88    1.196728
89    1.196728
90    1.196728
Name: color_out, Length: 91, dtype: float64

In [147]:
df['pricebook_unit_price_adjusted'] = (df['pricebook_unit_price'] * (1 - price_factor) * (1 + color_out_factor * (df['color_out'] != "WHT").astype(int))).round(2)
df['discrepancy'] = df['quote_unit_price'] - df['pricebook_unit_price_adjusted']
df[['quote_id', 'window_description', 'color_in', 'color_out', 'sq_ft',  'sq_ft_bucket', 'pricebook_unit_price', 'pricebook_unit_price_adjusted', 'quote_unit_price', 'discrepancy', 'source_file']]

,quote_id,window_description,color_in,color_out,sq_ft,sq_ft_bucket,pricebook_unit_price,pricebook_unit_price_adjusted,quote_unit_price,discrepancy,source_file
0,519307,CASEMENT,COLOR,COLOR,11.67,9-12,343.20,330.88,331.53,0.65,PO#_14FIELDING_Order_519307.PDF
1,519307,VINYL FIXED,COLOR,COLOR,11.67,7+,288.37,278.02,278.56,0.54,PO#_14FIELDING_Order_519307.PDF
2,519307,VINYL FIXED,COLOR,COLOR,29.00,7+,716.59,690.87,692.23,1.36,PO#_14FIELDING_Order_519307.PDF
3,519307,VINYL FIXED,COLOR,COLOR,11.67,7+,288.37,278.02,278.56,0.54,PO#_14FIELDING_Order_519307.PDF
4,519307,CASEMENT,COLOR,COLOR,11.67,9-12,343.20,330.88,331.53,0.65,PO#_14FIELDING_Order_519307.PDF
...,...,...,...,...,...,...,...,...,...,...,...
86,518692,CASEMENT,WHT,WHT,6.25,6-9,233.69,188.26,188.12,-0.14,PO#_STK101225_Order_518692.PDF
87,518692,CASEMENT,WHT,WHT,6.25,6-9,233.69,188.26,188.12,-0.14,PO#_STK101225_Order_518692.PDF
88,518692,CASEMENT,WHT,WHT,6.25,6-9,233.69,188.26,188.12,-0.14,PO#_STK101225_Order_518692.PDF
89,518692,CASEMENT,WHT,WHT,6.25,6-9,233.69,188.26,188.12,-0.14,PO#_STK101225_Order_518692.PDF


## Export to CSV
Write the DataFrame to `extracted_units.csv`.

In [134]:
df[['quote_id', 'window_description', 'color_in', 'color_out', 'sq_ft',  'sq_ft_bucket', 'pricebook_unit_price', 'pricebook_unit_price_adjusted', 'quote_unit_price', 'discrepancy', 'source_file']].to_csv(OUT_PATH, index=False)
print(f"Wrote {len(df)} rows to {OUT_PATH}")

Wrote 200 rows to extracted_units_pre2026.csv


## Distinct value counts
Counts per value for `color_in`, `color_out`, and `window_description`.

In [9]:
print("--- Distinct color_in (count) ---")
display(df["color_in"].value_counts().sort_index())

--- Distinct color_in (count) ---


color_in
               1
CHESTNUT B     6
WHT           97
Name: count, dtype: int64

In [10]:
print("--- Distinct color_out (count) ---")
display(df["color_out"].value_counts().sort_index())

--- Distinct color_out (count) ---


color_out
                    1
BLACK 525          59
PEBBLE/KAKI 559     3
WHT                41
Name: count, dtype: int64

In [11]:
print("--- Distinct window_description (count) ---")
display(df["window_description"].value_counts().sort_index())

--- Distinct window_description (count) ---


window_description
CASEMENT             48
DOUBLE END SLIDER     1
DOUBLE SLIDER         1
DUMMY WINDOW          2
HALF ROUND            7
SEALED UNIT           1
SINGLE SLIDER        13
SMALL FIXED           3
VINYL FIXED          28
Name: count, dtype: int64